# Exercice semaine 4

Using the usual Vahatra dataframe of protected areas (PAs) in Madagascar (containing all protected areas), perform the following tasks: 

1. Fit an OLS model with the deforestation rate over the period 2000 - 2023 as your response variable and predictors corresponding to a quadratic polynomial function of the year of creation of the protected area (year of creation + year of creation squared).
2. Print a summary of the results and compare the R-squared of this OLS model to the R-squared of the OLS model you fitted last week with the year of creation as your single predictor variable. What do you observe?
3. Next compare these two models using a F-test with the anova_lm() function. What do you observe?
4. Now create a dummy variable equal to 1 if a PA belongs to IUCN category I or II and to 0 otherwise. Fit an OLS model with this new variable as your predictor and the deforestation rate over the period 2000 - 2023 as the response variable. Print the summary table of results. What do you observe?
5. Finally, fit an OLS model with the year of creation and the IUCN category I or II dummy as your two predictors. Print the results and perform a F-test comparing this model to the model with the year of creation as the single predictor. What do you observe? Why do you think this is the case?


1. **Fit an OLS model with the deforestation rate over the period 2000 - 2023 as your response variable and predictors corresponding to a quadratic polynomial function of the year of creation of the protected area (year of creation + year of creation squared).**

In [2]:
pip install ISLP

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 65.9 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 103.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 62.1 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 97.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 92.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 96.9 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Install and load packages

import numpy as np #numerical operations
import pandas as pd #data wrangling

!pip install openpyxl #import Excel data
import openpyxl
 
import statsmodels.api as sm #regression models
from statsmodels.stats.outliers_influence \
     import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

from ISLP import load_data #matrix design and other regression related functions
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

from matplotlib.pyplot import subplots
import seaborn as sns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [5]:
#Data preparation
pas = pd.read_excel('Vahatra_defor_noNA.xlsx')
pas.columns = pas.columns.str.replace("-01-01_treecover_ha", "", regex=False)
pas.columns = pas.columns.str.replace("_2015-01-01_5k_110mio_traveltime_mean_minutes", "", regex=False)
pas['deforest_2000_2023'] = (pas['treecover_area_2000']-pas['treecover_area_2023'])/pas['treecover_area_2000']*100

In [6]:
# Check column names
pas.columns

Index(['nom', 'cat_iucn', 'creation', 'date_creation', 'date_modification',
       'mention_changement', 'hectares', 'num_atlas_', 'full_name', 'province',
       'region', 'district', 'gest_1', 'gest_2', 'type_ap', 'an_creation',
       'assetid', 'elevation_2000-02-01_elevation_mean_m',
       'slope_2000-02-01_slope_mean_degrees', 'treecover_area_2000',
       'treecover_area_2001', 'treecover_area_2002', 'treecover_area_2003',
       'treecover_area_2004', 'treecover_area_2005', 'treecover_area_2006',
       'treecover_area_2007', 'treecover_area_2008', 'treecover_area_2009',
       'treecover_area_2010', 'treecover_area_2011', 'treecover_area_2012',
       'treecover_area_2013', 'treecover_area_2014', 'treecover_area_2015',
       'treecover_area_2016', 'treecover_area_2017', 'treecover_area_2018',
       'treecover_area_2019', 'treecover_area_2020', 'treecover_area_2021',
       'treecover_area_2022', 'treecover_area_2023', 'traveltime', 'geometry',
       'Groupe', 'surface_m2',

In [7]:
#Matrix design
pas['year_creation'] = pas['date_creation'].dt.year
X = MS(['year_creation']).fit_transform(pas) # Matrix with single year of creation predictor
Z = MS([poly('year_creation', degree = 2, raw=True)]).fit_transform(pas) # Matrix year of creation + quadratic term as predictors
X

,intercept,year_creation
0,1.0,2015
1,1.0,2015
2,1.0,1958
3,1.0,1956
4,1.0,2015
...,...,...
92,1.0,2015
93,1.0,2015
94,1.0,2015
95,1.0,2015


In [8]:
Z

,intercept,"poly(year_creation, degree=2, raw=True)[0]","poly(year_creation, degree=2, raw=True)[1]"
0,1.0,2015.0,4060225.0
1,1.0,2015.0,4060225.0
2,1.0,1958.0,3833764.0
3,1.0,1956.0,3825936.0
4,1.0,2015.0,4060225.0
...,...,...,...
92,1.0,2015.0,4060225.0
93,1.0,2015.0,4060225.0
94,1.0,2015.0,4060225.0
95,1.0,2015.0,4060225.0


In [9]:
#Create response vector
y = pd.DataFrame(pas['deforest_2000_2023'])
y

,deforest_2000_2023
0,21.022159
1,13.385643
2,14.314523
3,42.940478
4,24.057451
...,...
92,38.989696
93,7.987383
94,12.981940
95,5.552914


In [11]:
# Fit quadratic model
model2 = sm.OLS(y, Z)
results2 = model2.fit()
results2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     deforest_2000_2023   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.026
Method:                 Least Squares   F-statistic:                     2.291
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.107
Time:                        17:32:50   Log-Likelihood:                -398.41
No. Observations:                  97   AIC:                             802.8
Df Residuals:                      94   BIC:                             810.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================================================
                                                 coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------
intercept                                   8050.3986   1.47e+04      0.546      0.586   -2.12e+04    3.73e+04
poly(year_creation, degree=2, raw=True)[0]    -8.2363     14.846     -0.555      0.580     -37.713      21.241
poly(year_creation, degree=2, raw=True)[1]     0.0021      0.004      0.564      0.574      -0.005       0.010
==============================================================================
Omnibus:                       32.067   Durbin-Watson:                   2.261
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               52.245
Skew:                           1.452   Prob(JB):                     4.52e-12
Kurtosis:                       5.121   Cond. No.                     3.88e+10
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.88e+10. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

2. **Print a summary of the results and compare the R-squared of this OLS model to the R-squared of the OLS model you fitted last week with the year of creation as your single predictor variable. What do you observe?**

In [14]:
# Fit single predictor model
model1 = sm.OLS(y, X)
results1 = model1.fit()
results1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     deforest_2000_2023   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     4.296
Date:                Tue, 03 Mar 2026   Prob (F-statistic):             0.0409
Time:                        17:41:19   Log-Likelihood:                -398.58
No. Observations:                  97   AIC:                             801.2
Df Residuals:                      95   BIC:                             806.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept      -259.2753    132.132     -1.962      0.053    -521.591       3.040
year_creation     0.1370      0.066      2.073      0.041       0.006       0.268
==============================================================================
Omnibus:                       33.103   Durbin-Watson:                   2.250
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               55.303
Skew:                           1.480   Prob(JB):                     9.80e-13
Kurtosis:                       5.217   Cond. No.                     1.75e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.75e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

The adjusted R-squared of the quadratic model is smaller than the R-squared and the adjusted R-squared of the simple model, suggesting that adding the quadratic term does not improve the fit of the model.

3.**Next compare these two models using a F-test with the anova_lm() function. What do you observe?**

In [13]:
anova_lm(results1, results2)

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,95.0,21055.012649,0.0,NaN,NaN,NaN
1,94.0,20983.998944,1.0,71.013706,0.318113,0.574087


The F-test comparing the two models confirms that the quadratic model does not explain a higher share of the variance of the response variable than the single-predictor model (p-value = 0.574).

4. **Now create a dummy variable equal to 1 if a PA belongs to IUCN category I or II and to 0 otherwise. Fit an OLS model with this new variable as your predictor and the deforestation rate over the period 2000 - 2023 as the response variable. Print the summary table of results. What do you observe?**

In [16]:
# Create dummy
pas['cat_iucn'] = pas['cat_iucn'].astype('category') # Convert cat_iucn to categorical
pas['cat_1_2'] = (pas['cat_iucn'] == 'I') | (pas['cat_iucn'] == 'II')
pas['cat_1_2']

0     False
1     False
2     False
3     False
4     False
      ...  
92    False
93    False
94    False
95    False
96    False
Name: cat_1_2, Length: 97, dtype: bool

In [17]:
# New matrix of predictors with category 1 or 2 dummy 
W = MS(['cat_1_2']).fit_transform(pas)
W.describe()

,intercept,cat_1_2
count,97.0,97.000000
mean,1.0,0.278351
std,0.0,0.450515
min,1.0,0.000000
25%,1.0,0.000000
50%,1.0,0.000000
75%,1.0,1.000000
max,1.0,1.000000


In [19]:
#Fit OLS model
model3 = sm.OLS(y, W)
results3 = model3.fit()
results3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     deforest_2000_2023   R-squared:                       0.035
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     3.414
Date:                Tue, 03 Mar 2026   Prob (F-statistic):             0.0677
Time:                        17:51:22   Log-Likelihood:                -399.01
No. Observations:                  97   AIC:                             802.0
Df Residuals:                      95   BIC:                             807.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     16.3034      1.787      9.122      0.000      12.755      19.852
cat_1_2       -6.2599      3.388     -1.848      0.068     -12.985       0.466
==============================================================================
Omnibus:                       31.632   Durbin-Watson:                   2.328
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               51.300
Skew:                           1.433   Prob(JB):                     7.25e-12
Kurtosis:                       5.118   Cond. No.                         2.44
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

IUCN category I or II protection status is negatively correlated with deforestation over 2000-2023. The deforestation rate is on average 6.3 percentage points lower in these protected areas compared to other protected areas (cat. III to VI or unclassified) over this period. The relationship is statistically significant at the 10-percent level (p-value = 0.068).

5. **Finally, fit an OLS model with the year of creation and the IUCN category I or II dummy as your two predictors. Print the results and perform a F-test comparing this model to the model with the year of creation as the single predictor. What do you observe? Why do you think this is the case?**

In [20]:
# Create matrix of predictors
R = MS(['year_creation', 'cat_1_2']).fit_transform(pas)

In [22]:
# Fit OLS model with year of creation and IUCN category dummy as predictors
model4 = sm.OLS(y, R)
results4 = model4.fit()
results4.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     deforest_2000_2023   R-squared:                       0.068
Model:                            OLS   Adj. R-squared:                  0.048
Method:                 Least Squares   F-statistic:                     3.428
Date:                Tue, 03 Mar 2026   Prob (F-statistic):             0.0366
Time:                        18:01:47   Log-Likelihood:                -397.31
No. Observations:                  97   AIC:                             800.6
Df Residuals:                      94   BIC:                             808.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept      -226.8204    132.708     -1.709      0.091    -490.314      36.674
year_creation     0.1215      0.066      1.832      0.070      -0.010       0.253
cat_1_2          -5.3426      3.384     -1.579      0.118     -12.061       1.376
==============================================================================
Omnibus:                       30.442   Durbin-Watson:                   2.303
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               48.083
Skew:                           1.396   Prob(JB):                     3.62e-11
Kurtosis:                       5.024   Cond. No.                     1.77e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.77e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [23]:
# F-test
anova_lm(results1, results4)

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,95.0,21055.012649,0.0,NaN,NaN,NaN
1,94.0,20511.039955,1.0,543.972694,2.492971,0.117715


The coefficient on the IUCN category dummy is no longer statistically significant when we also include the year of creation among the predictors. The p-value on the F-test and the p-value on the coefficient of the IUCN category in model 4 are equal. This is always the case when we compare two **OLS** models with only one additional constraint (predictor variable) added between the first and the second model. 